In [ ]:
Attribution:
AUTHOR: https://twitter.com/fchollet
WEBSITE: https://keras.io/examples/nlp/pretrained_word_embeddings/

In [43]:
import os

# Only the TensorFlow backend supports string inputs.
os.environ["KERAS_BACKEND"] = "tensorflow"

import pathlib
import numpy as np
import tensorflow.data as tf_data
import keras
from keras import layers
from ipywidgets import widgets

In [44]:
data_path = keras.utils.get_file(
         "news20.tar.gz",
         "http://www.cs.cmu.edu/afs/cs.cmu.edu/project/theo-20/www/data/news20.tar.gz",
         untar=True,
)

In [45]:
data_dir = pathlib.Path(data_path).parent / "20_newsgroup"
dirnames = os.listdir(data_dir)


fnames = os.listdir(data_dir / "comp.graphics")

In [46]:
# Load the data from the directories
samples = []
labels = []
class_names = []
class_index = 0

for dirname in sorted(os.listdir(data_dir)):
    class_names.append(dirname)
    dirpath = data_dir / dirname
    fnames = os.listdir(dirpath)
    print("Processing %s, %d files found" % (dirname, len(fnames)))
    
    for fname in fnames:
        fpath = dirpath / fname
        f = open(fpath, encoding="latin-1")
        content = f.read()
        lines = content.split("\\n")
        lines = lines[10:]
        content = "\\n".join(lines)
        samples.append(content)
        labels.append(class_index)
    
    class_index += 1


Processing alt.atheism, 1000 files found
Processing comp.graphics, 1000 files found
Processing comp.os.ms-windows.misc, 1000 files found
Processing comp.sys.ibm.pc.hardware, 1000 files found
Processing comp.sys.mac.hardware, 1000 files found
Processing comp.windows.x, 1000 files found
Processing misc.forsale, 357 files found


In [47]:
# Shuffle and split the data into training & validation sets
seed = 1337
rng = np.random.RandomState(seed)
rng.shuffle(samples)
rng = np.random.RandomState(seed)
rng.shuffle(labels)

# Extract a training & validation split
validation_split = 0.2
num_validation_samples = int(validation_split * len(samples))
train_samples = samples[:-num_validation_samples]
val_samples = samples[-num_validation_samples:]
train_labels = labels[:-num_validation_samples]
val_labels = labels[-num_validation_samples:]

In [48]:
# Create a vocabulary index
vectorizer = layers.TextVectorization(max_tokens=20000, output_sequence_length=200)
text_ds = tf_data.Dataset.from_tensor_slices(train_samples).batch(128)
vectorizer.adapt(text_ds)

# Retrieve the computed vocabulary
voc = vectorizer.get_vocabulary()
word_index = dict(zip(voc, range(len(voc))))

# Mapping words to their indices
voc = vectorizer.get_vocabulary()
word_index = dict(zip(voc, range(len(voc))))



In [49]:
dropdown = widgets.Dropdown(
    options=[('50d', './glove.6B.50d.txt'), ('100d', './glove.6B.100d.txt'), ('200d', './glove.6B.200d.txt'), ('300d', './glove.6B.300d.txt')],
    value='./glove.6B.50d.txt',
    description='GLOVE file:',
)
display(dropdown)

Dropdown(description='GLOVE file:', options=(('50d', './glove.6B.50d.txt'), ('100d', './glove.6B.100d.txt'), (…

In [50]:
path_to_glove_file = f"{dropdown.value}"

embeddings_index = {}
with open(path_to_glove_file, encoding='utf-8') as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print("Found %s word vectors." % len(embeddings_index))

Found 400000 word vectors.


In [51]:
path_list = path_to_glove_file.split(".")
embedding_dim = 0
for item in path_list:
    if "d" in item:
        embedding_dim = int(item[:-1])
        break

In [52]:
num_tokens = len(voc) + 2
hits = 0
misses = 0

# Prepare embedding matrix
embedding_matrix = np.zeros((num_tokens, embedding_dim))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # Words not found in embedding index will be all-zeros.
        # This includes the representation for "padding" and "OOV"
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        misses += 1
print("Converted %d words (%d misses)" % (hits, misses))


Converted 717 words (392 misses)


In [53]:
from keras.layers import Embedding

embedding_layer = Embedding(
    num_tokens,
    embedding_dim,
    trainable=False,
)
embedding_layer.build((1,))
embedding_layer.set_weights([embedding_matrix])


In [54]:
int_sequences_input = keras.Input(shape=(None,), dtype="int32")
embedded_sequences = embedding_layer(int_sequences_input)
x = layers.Conv1D(128, 5, activation="relu")(embedded_sequences)
x = layers.MaxPooling1D(5)(x)
x = layers.Conv1D(128, 5, activation="relu")(x)
x = layers.MaxPooling1D(5)(x)
x = layers.Conv1D(128, 5, activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)
preds = layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(int_sequences_input, preds)
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, None)]            0         
                                                                 
 embedding_1 (Embedding)     (None, None, 50)          55550     
                                                                 
 conv1d_3 (Conv1D)           (None, None, 128)         32128     
                                                                 
 max_pooling1d_2 (MaxPooling  (None, None, 128)        0         
 1D)                                                             
                                                                 
 conv1d_4 (Conv1D)           (None, None, 128)         82048     
                                                                 
 max_pooling1d_3 (MaxPooling  (None, None, 128)        0         
 1D)                                                       

In [55]:
# Train the model
x_train = vectorizer(np.array([[s] for s in train_samples])).numpy()
x_val = vectorizer(np.array([[s] for s in val_samples])).numpy()

y_train = np.array(train_labels)
y_val = np.array(val_labels)

In [56]:
# Use categorical crossentropy as our loss since we're doing softmax classification
model.compile(loss="sparse_categorical_crossentropy", optimizer="rmsprop", metrics=["acc"])

In [57]:
# Train the model
model.fit(x_train, y_train, batch_size=128, epochs=20, validation_data=(x_val, y_val))

Epoch 1/20
40/40 [==============================] - 2s 44ms/step - loss: 1.9250 - acc: 0.1455 - val_loss: 1.9102 - val_acc: 0.1597
Epoch 2/20
40/40 [==============================] - 2s 40ms/step - loss: 1.9141 - acc: 0.1597 - val_loss: 1.9134 - val_acc: 0.1542
Epoch 3/20
40/40 [==============================] - 2s 40ms/step - loss: 1.9143 - acc: 0.1496 - val_loss: 1.9101 - val_acc: 0.1589
Epoch 4/20
40/40 [==============================] - 2s 40ms/step - loss: 1.9120 - acc: 0.1512 - val_loss: 1.9108 - val_acc: 0.1542
Epoch 5/20
40/40 [==============================] - 2s 43ms/step - loss: 1.9097 - acc: 0.1524 - val_loss: 1.9104 - val_acc: 0.1597
Epoch 6/20
40/40 [==============================] - 2s 41ms/step - loss: 1.9077 - acc: 0.1612 - val_loss: 1.9106 - val_acc: 0.1605
Epoch 7/20
40/40 [==============================] - 2s 41ms/step - loss: 1.9078 - acc: 0.1718 - val_loss: 1.9113 - val_acc: 0.1471
Epoch 8/20
40/40 [==============================] - 2s 42ms/step - loss: 1.9099 - a

In [58]:
model.save("glove-newsgroups")

INFO:tensorflow:Assets written to: glove-newsgroups\assets


INFO:tensorflow:Assets written to: glove-newsgroups\assets
